In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as funcy

In [2]:
class Ex(nn.Module):
    def __init__(self,dm,dh):
        super().__init__()
        self.l1=nn.Linear(dm,dh)
        self.l2=nn.Linear(dh,dm)
    def forward(self,x):
        return self.l2(funcy.relu(self.l1(x)))

In [3]:
class COS(nn.Module):
    def __init__(self,dm,dh,exp=5,top_k=4):
        super().__init__()
        self.exp=exp
        self.top=top_k
        self.exps=nn.ModuleList([Ex(dm,dh) for _ in range(exp)])
        self.e=nn.Parameter(torch.randn(exp,dm))
    def forward(self,x):
        bs,sl,dm=x.shape
        xf=x.view(-1,dm)
        xn=funcy.normalize(xf,dim=-1)
        en=funcy.normalize(self.e,dim=-1)
        cos=torch.matmul(xn,en.T)
        t=0.1
        cos=cos/t
        probs=funcy.softmax(cos,dim=-1)
        tv,ti=torch.topk(probs,self.top,dim=-1)
        out=torch.zeros_like(xf)
        for ei in range(self.exp):
            toi,si=(ti==ei).nonzero(as_tuple=True)
            if len(toi)==0:
                continue
            sx=xf[toi]
            wei=tv[toi,si]
            eout=self.exps[ei](sx)
            out[toi]+=wei.unsqueeze(-1)*eout
        out=out.view(bs,sl,dm)
        return out

In [4]:
moe=COS(32,64,5,4)

In [5]:
x = torch.randn(2, 5, 32)
y = moe(x)

print(y.shape)

torch.Size([2, 5, 32])
